# Phase 9 v2 — CNN Model: Adding Arecanut + Cocoa Black Pod
## AI-Based Crop Disease Detection and Smart Farming Guidance System — Arecanut belt (Dakshina Kannada / Udupi / Shivamogga) mixed-cropping system

**Run this in Google Colab** with a GPU runtime: `Runtime > Change runtime type > T4 GPU`.

**What changed from v1:** `model_v1.h5` covers 7 classes — 3 Black Pepper diseases + 4 Rice diseases.
Cocoa was *intended* for v1 (see `ml-training/README.md`) but didn't make it into the final run, and
Arecanut was left out entirely because no public image dataset existed for it at the time
(`PROJECT_SCOPE_PIVOT.md`'s dataset survey). Both of those are now addressable:

- **Arecanut**: a public dataset now exists — [`BeenaPradeep/ARECANUT-DATASET`](https://github.com/BeenaPradeep/ARECANUT-DATASET)
  on GitHub (~300 field images, 6 classes). This notebook adds **2 of your 3 seeded Arecanut
  diseases** from it: Yellow Leaf Disease (YLD) and Mahali/Koleroga (Bud Rot). **Foot Rot (Anabe
  Roga) still has no matching data in this dataset — it stays uncovered.**
- **Cocoa**: neither public Cocoa dataset found in the scope survey actually has classes named
  "Black Pod" or "VSD". The closest usable match is `bryandarquea/cocoa-diseases`'s `phytophthora`
  class — *Phytophthora palmivora* is the literal causal agent of Black Pod Disease, so this is a
  legitimate relabel, not a stretch. **VSD has no public data anywhere found — it stays uncovered.**

**This is a full retrain, not an incremental add** — MobileNetV2's classification head is a fixed-size
softmax, so v2 trains from scratch on the *union* of all classes (Pepper ×3, Rice ×4, Arecanut ×2,
Cocoa ×1 = **10 classes**), not just the 3 new ones. v1's Pepper/Rice coverage carries forward
unchanged in v2, just retrained together with the new classes.

**What this notebook does, in order:** same 8-step pipeline as v1 (download → inspect & map →
split → augment → MobileNetV2 transfer learning → train → fine-tune → evaluate → save), with one
extra download source (GitHub, for Arecanut) alongside the three existing Kaggle ones.

## 1. Setup

Colab already has TensorFlow. We only need a few extra packages for the metrics/plots and for a clean train/val/test split.

In [ ]:
!pip install -q split-folders scikit-learn seaborn

import os, json, shutil, random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print("\nWARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
          "then Runtime > Restart session, and re-run from the top. Training will be "
          "very slow on CPU.")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


## 2. Download the datasets — Rice, Black Pepper, Cocoa (Kaggle) + Arecanut (GitHub)

Rice/Pepper/Cocoa still come from Kaggle, same as v1. Arecanut comes from a public GitHub repo
instead — no Kaggle account needed for that one, just `git clone`. Both land in the same
`kaggle_raw/<Crop>/` layout so the rest of the pipeline (folder inspection, class mapping,
splitting) treats all four crops identically regardless of where the images came from.

In [ ]:
import getpass, os

kaggle_token = getpass.getpass("Paste your Kaggle API token (KGAT_...): ")
os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

!pip install -q "kaggle>=1.8.0"
!kaggle datasets list -s rice > /dev/null && print("Kaggle API ready — token works.")


Now download the three Kaggle datasets. Same handles as v1 — chosen from the dataset survey in
`PROJECT_SCOPE_PIVOT.md` for being the most complete/usable option for that crop.

In [ ]:
DATASETS = {
    "Rice":  "nirmalsankalana/rice-leaf-disease-image",
    "Pepper": "adithyantg/black-pepper-leaf-disease-mini-dataset",
    "Cocoa": "bryandarquea/cocoa-diseases",
}

RAW_DOWNLOAD_ROOT = "kaggle_raw"
shutil.rmtree(RAW_DOWNLOAD_ROOT, ignore_errors=True)
os.makedirs(RAW_DOWNLOAD_ROOT, exist_ok=True)

for crop, handle in DATASETS.items():
    dest = os.path.join(RAW_DOWNLOAD_ROOT, crop)
    os.makedirs(dest, exist_ok=True)
    print(f"\n=== Downloading {crop}: {handle} ===")
    !kaggle datasets download -d {handle} -p {dest} --unzip


Now clone the Arecanut dataset from GitHub (not Kaggle — a plain public repo, no auth needed).
If this specific URL has moved or been renamed by the time you run this, search GitHub for
"arecanut disease dataset" or check `IEEE DataPort` (same dataset, same 6 classes, but requires a
subscription) as a fallback.

In [ ]:
ARECANUT_REPO = "https://github.com/BeenaPradeep/ARECANUT-DATASET.git"

arecanut_dest = os.path.join(RAW_DOWNLOAD_ROOT, "Arecanut")
shutil.rmtree(arecanut_dest, ignore_errors=True)
print(f"\n=== Cloning Arecanut: {ARECANUT_REPO} ===")
!git clone --depth 1 {ARECANUT_REPO} {arecanut_dest}

# Drop the .git folder so it doesn't get walked/counted as "images" below.
shutil.rmtree(os.path.join(arecanut_dest, ".git"), ignore_errors=True)

ALL_CROPS = list(DATASETS) + ["Arecanut"]
print("\nAll crop sources ready:", ALL_CROPS)


### Inspect the real folder structure

This prints every folder that directly contains image files, for each of the four crops, with how
many images are in it. **Read this output carefully before the next cell** — the exact names depend
on how each dataset was organized (could be `train/Blast/`, `Healthy/`, `bacterial_leaf_blight/`,
anything). The Arecanut repo's class names are a best guess from its README (`Bud Rot`,
`Healthy Leaf`, `Healthy Nut`, `Koleroga`, `Ring Spot`, `Yellow Leaf`) — confirm against what's
actually printed below, not this description.

In [ ]:
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG")

def find_image_dirs(root, base):
    """Return every directory under `root` that directly contains image files, with counts.
    Paths are reported relative to `base` (so they include the crop-name prefix and can be
    pasted straight into CLASS_FOLDER_MAP below)."""
    found = []
    for dirpath, _dirnames, filenames in os.walk(root):
        images = [f for f in filenames if f.endswith(IMAGE_EXTS)]
        if images:
            rel = os.path.relpath(dirpath, base)
            found.append((rel, len(images)))
    return sorted(found)

for crop in ALL_CROPS:
    print(f"\n=== {crop} — folders containing images ===")
    for rel, n in find_image_dirs(os.path.join(RAW_DOWNLOAD_ROOT, crop), RAW_DOWNLOAD_ROOT):
        print(f"  {rel!r}: {n} images")


### Fill in the class mapping

Copy the exact folder paths printed above (the `'...'` strings) into `CLASS_FOLDER_MAP` below, one
entry per line, mapped to a clean class label in the `Crop___Condition` format (matching v1's style
and the disease names already seeded in the database — check
`backend/crops/management/commands/seed_demo_data.py` if you want to confirm exact names).

**Only map classes you actually want trained — don't feel obliged to use every folder a dataset
offers.** Specifically for this v2 run:
- From **Arecanut**: map `Yellow Leaf` → `Arecanut___Yellow_Leaf_Disease` and `Koleroga` and/or
  `Bud Rot` → `Arecanut___Mahali_Koleroga` (these two names refer to the same disease in Kannada
  agricultural usage — if the dataset has both as separate folders, you can map both to the same
  class label here, they'll just get combined). Skip `Ring Spot` (no matching disease in the DB
  yet) and the `Healthy Leaf`/`Healthy Nut` folders (v1's Pepper/Rice classes are disease-only, so
  this run keeps that same convention — healthy-class support would be a good follow-up, but is a
  bigger design decision for the whole model, not just Arecanut).
- From **Cocoa**: map the `phytophthora` class → `Cocoa___Black_Pod_Disease`. Skip `monilia` (a
  different disease — frosty pod rot — not in the seeded DB) and `healthy`, same reasoning as above.
- **Rice and Pepper**: map exactly like v1 did (see `training_report.md` for the 7 class names v1
  used, if you want to match them exactly).

If a dataset has a `Healthy` folder you *do* want to include anyway, that's a reasonable call too —
just know it changes the model's design (adds a "none of the above" option) and you'll want to
re-check `scans/ml_model.py`'s `LABEL_TO_DISEASE` table expects exactly the classes you end up with.

In [ ]:
CLASS_FOLDER_MAP = {
    # Example — replace these with the REAL paths from the printed output above:
    # "Rice/train/Bacterialblight":      "Rice___Bacterial_Leaf_Blight",
    # "Rice/train/Blast":                "Rice___Blast",
    # "Pepper/Foot_Rot":                 "Pepper___Foot_Rot",
    # "Cocoa/phytophthora":              "Cocoa___Black_Pod_Disease",
    # "Arecanut/Yellow Leaf":            "Arecanut___Yellow_Leaf_Disease",
    # "Arecanut/Koleroga":               "Arecanut___Mahali_Koleroga",
    # "Arecanut/Bud Rot":                "Arecanut___Mahali_Koleroga",
}

assert CLASS_FOLDER_MAP, (
    "CLASS_FOLDER_MAP is empty — fill it in using the folder paths printed in the cell above "
    "before continuing."
)
print(f"{len(CLASS_FOLDER_MAP)} classes mapped:")
for k, v in CLASS_FOLDER_MAP.items():
    print(f"  {k!r} -> {v}")


In [ ]:
# Copy a capped number of images per mapped class into a clean, flat dataset_raw/ folder
MAX_IMAGES_PER_CLASS = 400  # caps class size for a faster, still-real first training run

DATASET_ROOT = "dataset_raw"
shutil.rmtree(DATASET_ROOT, ignore_errors=True)
os.makedirs(DATASET_ROOT, exist_ok=True)

class_counts = {}
for rel_path, label in CLASS_FOLDER_MAP.items():
    # rel_path is relative to RAW_DOWNLOAD_ROOT, exactly as printed above (crop-name prefix included).
    src_dir = os.path.join(RAW_DOWNLOAD_ROOT, rel_path)
    if not os.path.isdir(src_dir):
        raise FileNotFoundError(
            f"{src_dir!r} does not exist — double-check the path you copied from the printed tree above."
        )
    files = sorted(f for f in os.listdir(src_dir) if f.endswith(IMAGE_EXTS))
    random.shuffle(files)
    files = files[:MAX_IMAGES_PER_CLASS]
    dst_dir = os.path.join(DATASET_ROOT, label)
    os.makedirs(dst_dir, exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))
    # A class mapped from two source folders (e.g. Koleroga + Bud Rot) accumulates here —
    # report the running total, not just this folder's count.
    class_counts[label] = class_counts.get(label, 0) + len(files)

print("Images per class (after capping):")
for cls, n in class_counts.items():
    print(f"  {cls}: {n}")
print(f"\nTotal images: {sum(class_counts.values())}")

if any(n < 30 for n in class_counts.values()):
    print("\nWARNING: at least one class has under 30 images — expect noisy precision/recall for "
          "that class specifically. Report it honestly rather than hiding it; small-class disease "
          "data is a known, real limitation (see the Arecanut note in PROJECT_SCOPE_PIVOT.md).")


## 3. Train / validation / test split

70% train, 15% validation, 15% test, stratified per class by `split-folders`.

In [ ]:
import splitfolders

SPLIT_ROOT = "dataset_split"
shutil.rmtree(SPLIT_ROOT, ignore_errors=True)
splitfolders.ratio(DATASET_ROOT, output=SPLIT_ROOT, seed=SEED, ratio=(0.7, 0.15, 0.15))

for split in ["train", "val", "test"]:
    n = sum(len(files) for _, _, files in os.walk(os.path.join(SPLIT_ROOT, split)))
    print(f"{split}: {n} images")


## 4. Data pipeline with augmentation

Augmentation (rotation, flip, zoom) is applied only to the training set — validation and test data
stay unmodified so evaluation reflects real, unaltered images.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/train", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", seed=SEED
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/val", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", seed=SEED
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{SPLIT_ROOT}/test", image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical", seed=SEED, shuffle=False
)

class_names = train_ds.class_names
num_classes = len(class_names)
print(f"{num_classes} classes:", class_names)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
], name="augmentation")

AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input

def prep_train(x, y):
    x = data_augmentation(x)
    x = preprocess(x)
    return x, y

def prep_eval(x, y):
    x = preprocess(x)
    return x, y

train_ds_p = train_ds.map(prep_train, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds_p = val_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds_p = test_ds.map(prep_eval, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


## 5. Build the model — MobileNetV2 transfer learning

The pretrained base is frozen first (train only the new head), then optionally fine-tuned in the next
section. Freezing first prevents the randomly-initialized head from wrecking the pretrained weights
with large early gradients.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 6. Train the head (base frozen)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
]

EPOCHS = 15
history = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=EPOCHS,
    callbacks=callbacks,
)


## 7. Fine-tune (optional but recommended)

Unfreeze the top layers of the MobileNetV2 base and continue training at a much lower learning rate.
This usually improves accuracy meaningfully over the frozen-base run — worth doing for the number you
actually report to your guide, but the frozen-base run above is already a legitimate first result if
you're short on time.

In [ ]:
base_model.trainable = True

# Freeze everything except the last ~30 layers of the base network.
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

FT_EPOCHS = 10
history_fine = model.fit(
    train_ds_p,
    validation_data=val_ds_p,
    epochs=FT_EPOCHS,
    callbacks=callbacks,
)


## 8. Evaluate on the held-out test set

This is the number that matters — the model has never seen these images, augmented or otherwise. With
10 classes now instead of 7, and two of them (Arecanut, Cocoa Black Pod) backed by much smaller
datasets than Rice/Pepper, expect per-class precision/recall to vary more than v1's report did —
report it exactly as it comes out, the same honesty standard v1 used.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds_p)
print(f"\nTest accuracy: {test_accuracy:.4f}")
print(f"Test loss: {test_loss:.4f}")

y_true = []
y_pred = []
for images, labels in test_ds_p:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n=== Classification Report (real numbers from this run) ===")
report = classification_report(y_true, y_pred, target_names=class_names, digits=3)
print(report)

with open("training_report.md", "w") as f:
    f.write("# Phase 9 v2 — Training Report\n\n")
    f.write("Adds Arecanut (Yellow Leaf Disease, Mahali/Koleroga) and Cocoa Black Pod Disease to "
            "v1's Pepper + Rice coverage. Foot Rot (Arecanut) and VSD (Cocoa) remain uncovered — "
            "no public dataset found for either.\n\n")
    f.write(f"Test accuracy: {test_accuracy:.4f}\n\n")
    f.write(f"Test loss: {test_loss:.4f}\n\n")
    f.write("## Classification Report\n\n```\n" + report + "\n```\n")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(max(8, num_classes * 0.6), max(6, num_classes * 0.5)))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Phase 9 v2 Retrain (+ Arecanut + Cocoa Black Pod)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()


## 9. Save the model and label mapping

Versioned as `v2` so it sits alongside `model_v1.h5`/`labels_v1.json` rather than overwriting them —
if v2's numbers for the new small classes turn out rough, v1 is still there as a known-good fallback
for the Pepper/Rice demo while you iterate.

In [ ]:
MODEL_VERSION = "v2"

model.save(f"model_{MODEL_VERSION}.h5")

label_map = {i: name for i, name in enumerate(class_names)}
with open(f"labels_{MODEL_VERSION}.json", "w") as f:
    json.dump(label_map, f, indent=2)

print(f"Saved model_{MODEL_VERSION}.h5 and labels_{MODEL_VERSION}.json")
print("Label mapping:", label_map)


## 10. Download your results

Run the cell below to download the four files this notebook produces. Put all four into your
project's `ml-training/` folder (next to `model_v1.h5`/`labels_v1.json` — don't overwrite them,
this is a separate version) — then let Claude know they're there so `scans/ml_model.py` and
`LABEL_TO_DISEASE` can be updated to load `model_v2.h5` and wire up the 3 new disease classes.

In [ ]:
from google.colab import files

for fname in ["model_v2.h5", "labels_v2.json", "training_report.md", "confusion_matrix.png"]:
    if os.path.exists(fname):
        files.download(fname)
    else:
        print(f"Missing: {fname} — check the cells above ran without error.")


---
## Notes for your guide meeting

- **This is still an honest partial fix, not a complete one.** Of the project's 11 seeded diseases
  across 4 crops, this v2 run covers 8: all 3 Pepper diseases, all 4 Rice diseases, and now 2 of
  Arecanut's 3 (Yellow Leaf Disease, Mahali/Koleroga) plus 1 of Cocoa's 2 (Black Pod Disease).
  **Arecanut's Foot Rot and Cocoa's VSD have no public image data anywhere found** (checked as of
  October 2026) — they stay guidance-only (lifecycle/fertilizer/organic-manure content, no CNN
  detection) until that changes.
- **Arecanut's data is small** (~300 images across 6 original classes, split across however many you
  mapped) — expect its precision/recall to be noisier than Rice/Pepper's, which have more images per
  class. Report `training_report.md`'s real numbers, including if a class underperforms — that's a
  legitimate, citable observation about data scarcity in underserved regional crops, not a flaw to
  hide.
- **The Cocoa Black Pod class is relabeled from a dataset's `phytophthora` class**, not explicitly
  named "Black Pod" by its uploader — worth stating that provenance explicitly in your report/paper
  rather than implying the dataset was purpose-built for this.
- This is still a genuine multi-crop model evaluated as one coherent system (10 classes now instead
  of 7), consistent with the framing `PROJECT_SCOPE_PIVOT.md` recommended for v1.
- Once `model_v2.h5` and `labels_v2.json` are back in `ml-training/`, the next step is updating
  `backend/scans/ml_model.py` to load v2 instead of v1 and extending `LABEL_TO_DISEASE` with the 3
  new classes — a quick follow-up once these files exist, not done yet since they don't exist until
  you run this.